# MERT / EAT full fine-tuning — H100 × 1, 7개 실험

**제작 상태: GPU 학습 미실행.** 노트북 한 파일에 분할 메타데이터와 실행 코드를 포함했습니다. 원본 음원과 pretrained 가중치는 포함하지 않습니다.

한 GPU에서 독립 실험 01~07을 순차 실행합니다. HTDemucs는 고정하며, head는 기존 C08의 64채널·3층 residual Conv1D + max pooling + 64→32→2 MLP로 통일합니다. 원본을 먼저 분할하고 5초 구간을 생성합니다.

| 번호 | 분할 | encoder | 학습 |
|---|---|---|---|
| 01 | S0 기존 분할 | MERT330 | head만 |
| 02 | S0 | MERT330 | 전체 encoder + head |
| 03 | S0 | EAT-large | 전체 encoder + head |
| 04 | S1 MusicGen 제외 | MERT330 | 전체 encoder + head |
| 05 | S1 MusicGen 제외 | EAT-large | 전체 encoder + head |
| 06 | S2 AudioLDM 제외 | MERT330 | 전체 encoder + head |
| 07 | S2 AudioLDM 제외 | EAT-large | 전체 encoder + head |

S0의 기존 test는 여러 번 사용한 **과거 평가셋**입니다. S1·S2도 새 음원이 아니라 기존 train을 재분할한 스트레스 평가입니다. 새 미사용 외부 holdout이라고 해석하지 마세요. 각 분할 내부의 원본/PCM 중복은 검사하지만, 재인코딩된 유사 음원까지 모두 탐지했다는 뜻은 아닙니다.

## 1. 입력과 실행 방법

VESSL H100 80GB 한 장, Python 3.11 환경을 사용합니다. `dacon-v6-training` Object Storage를 `/shared`에 마운트하고 이 노트북을 `/root` 등 쓰기 가능한 작업 디렉터리로 복사합니다. `PROJECT_ROOT`와 `OUTPUT`은 충분한 공간이 있는 영구 볼륨에 두세요.

- 원본 음원 1,356개: 기존 `v6/V6_H100_RunBundle.zip` 및 `c3/full-final-20260915/C3_Final_Input.zip` 재사용.
- 모델·입력 목록: `/shared/fullft7/h100x1-20260916/`. MERT/EAT/HTDemucs는 로컬 경로로만 읽으며 Hub 다운로드를 하지 않습니다.
- 실행 시에만 `RUN_STAGE_INPUTS=True`로 두 ZIP에서 필요한 음원을 추출합니다. SHA-256이 기존 분할 기록과 같아야 합니다.
- 의존성 설치가 필요하면 `INSTALL_DEPENDENCIES=True`로 설치 셀만 실행하고 커널을 재시작합니다. 라이브러리 설치에는 네트워크가 필요합니다.
- 실제 실험 시 `RUN_PREPARE=True`, `RUN_TRAIN=True`로 설정합니다. HTDemucs 파형 캐시를 만든 후 실험 01~07을 한 개씩 실행하며, 중단 후에는 마지막 완료 epoch부터 재개합니다.
- `RUN_CHECKS`를 포함한 모든 실행 플래그는 기본 False입니다. 이번 수정 후 테스트·학습은 실행하지 않았습니다.

작업 종료 후 **VESSL 콘솔에서 리소스를 Stop/Pause**해야 컴퓨트 과금이 끝납니다. 노트북은 리소스를 자동 종료하지 않습니다. GPU 호환성과 실제 처리량은 아직 확인하지 않았습니다.


In [ ]:
from pathlib import Path
import time

STORAGE_ROOT = Path('/shared')
ASSET_ROOT = STORAGE_ROOT / 'fullft7/h100x1-20260916'
PROJECT_ROOT = Path.cwd()  # 쓰기 가능한 영구 볼륨: 추출한 v4_validation/audio 저장
NOTEBOOK_PATH = PROJECT_ROOT / 'MERT_EAT_FullFT_H100x1_7Experiments.ipynb'
OUTPUT = PROJECT_ROOT / 'h100x1_fullft7_run'
INSTALL_DEPENDENCIES = False
RUN_STAGE_INPUTS = False
RUN_CHECKS = False
RUN_PREPARE = False
RUN_TRAIN = False

# 계획 한도이며 실제 계정 잔액 또는 과금 상한을 보증하지 않습니다.
CREDITS_PER_HOUR = 2.98  # H100 × 1 Cloud 공개 단가; 실제 콘솔 표시값 우선
MAX_SESSION_CREDITS = 48.0
MAX_SESSION_HOURS = 16.0
SESSION_STARTED_AT = None  # 실제 리소스 대여 시작 Unix timestamp를 알면 입력

TRAIN_CONFIG = dict(seed=41, seconds=5.0, pool_windows=8, train_windows=4,
    epochs=20, patience=3, encoder_lr=1e-5, head_lr=1e-4,
    weight_decay=.01, warmup_fraction=.05, accumulation=8)


## 2. 크레딧 계산

[VESSL Cloud 공개 가격표](https://vessl.ai/en/pricing)의 H100 한 장 단가 **2.98 credits/h**를 사용합니다(1 credit = USD 1). 실제 콘솔 단가가 다르면 설정을 수정하세요. GPU 사용료만 계산하며 세금·저장공간·플랜 비용은 제외합니다.

아래 시간은 기존 H100 두 장의 미측정 추정을 한 장의 순차 실행으로 단순 환산한 계획치입니다. 실측 처리량이 없으므로 소요시간을 확정할 수 없습니다. 1 seed, 최대 20 epoch, 5초 입력, 분리 캐시 재사용 조건이며, 새 HTDemucs 캐시 생성과 초기 파일 읽기·설치는 추가됩니다. 48크레딧 안에 7개가 모두 끝난다는 보장은 없습니다.


In [ ]:
for label, low, high in [('기본 3개',4,8),('S1 포함 5개',6,12),('S2 포함 7개',8,16)]:
    print(f'{label}: 계획치 {low}~{high}시간 / {low*CREDITS_PER_HOUR:.2f}~{high*CREDITS_PER_HOUR:.2f} credits')
print(f'한도: {MAX_SESSION_CREDITS:.2f} credits, 최대 {MAX_SESSION_HOURS:.1f}시간')
print('H100 한 장을 6시간 사용하면 6 GPU·시간입니다.')


## 3. 환경 설치

Python 3.11 기준입니다. `INSTALL_DEPENDENCIES=True`일 때만 아래 설치가 실행됩니다. 설치 후 커널을 재시작하고 False로 돌려놓으세요. CUDA 라이브러리 설치에는 네트워크가 필요합니다. MERT/EAT/HTDemucs 가중치와 모델 코드는 업로드한 로컬 파일에서만 읽습니다.


In [ ]:
import subprocess
import sys
import shutil
if INSTALL_DEPENDENCIES:
    if shutil.which('uv') is None:
        raise RuntimeError('먼저 VESSL 터미널에서 uv를 설치한 뒤 이 셀을 실행하세요: https://docs.astral.sh/uv/getting-started/installation/')
    subprocess.run(['uv','pip','install','--python',sys.executable,
        'torch==2.7.1','torchaudio==2.7.1','torchvision==0.22.1',
        '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
    subprocess.run(['uv','pip','install','--python',sys.executable,
        'transformers==4.51.3','timm==1.0.15','numpy==1.26.4','soundfile==0.13.1',
        'demucs==4.0.1','einops==0.8.1','nbformat==5.10.4','nnAudio==0.3.3'],check=True)
    print('설치 완료. 커널을 재시작하고 INSTALL_DEPENDENCIES=False로 바꾸세요.')

## 4. 분할과 평가 규칙

S0는 기존 locked split을 유지하되 **NSynth를 모든 실험에서 제외**합니다. 이 선택은 모델 비교 전에 고정하며 별도의 NSynth 실험은 하지 않습니다. P0도 새 5초/고정 구간 정책으로 다시 학습하는 대조군이므로 기존 제출 점수 재현 실험과는 다릅니다.

S1·S2는 S0 train의 원본/아티스트 연결 그룹을 해시 순서로 약 60/20/20 분할합니다. 학습·tune에서는 MusicGen 또는 AudioLDM의 TTA·ATA를 모두 제외합니다. test에는 다른 생성기도 남겨 seen/held-generator 결과를 함께 출력합니다. 원본 그룹 배치는 두 분할에서 같습니다. **제외 생성기의 test 점수는 checkpoint 선택에 쓰지 않습니다.**

기존 train에서 특정 생성기와 원본을 공유하는 자료를 전부 제거하는 방식은 MusicGen/AudioLDM에서 학습 자료가 사라지므로 사용하지 않습니다. 생성기 제외 실험마다 pretrained 초기값부터 다시 학습합니다. S0 체크포인트를 S1·S2에 이어 학습하지 않습니다.

학습 샘플러는 real/fake에 같은 질량을 주고 fake 안에서 생성기 비중을 맞춥니다. tune/test에는 샘플링을 적용하지 않습니다.

In [ ]:
import base64
import collections
import hashlib
import json
import math
import os
from pathlib import Path
import random
import time
import zlib
from dataclasses import dataclass
from itertools import groupby
from typing import Literal, TypedDict

class AudioRow(TypedDict):
    ID: str
    path: str
    group_id: str
    source: str
    panel: str
    MUSIC_FAKE: str
    MUSIC_PRESENT: str
    FILE_FAKE: str
    VOICE_FAKE: str
    VOICE_PRESENT: str
    audio_sha256: str
    pcm_sha256: str

class Partition(TypedDict):
    train: list[AudioRow]
    tune: list[AudioRow]
    test: list[AudioRow]

@dataclass(frozen=True, slots=True)
class Trial:
    name: str
    split: str
    encoder: Literal['mert', 'eat']
    frozen: bool = False

TRIALS = [Trial('01_S0_MERT_frozen', 'S0', 'mert', True),
          Trial('02_S0_MERT_fullft', 'S0', 'mert'),
          Trial('03_S0_EAT_fullft', 'S0', 'eat'),
          Trial('04_S1_MERT_fullft', 'S1', 'mert'),
          Trial('05_S1_EAT_fullft', 'S1', 'eat'),
          Trial('06_S2_MERT_fullft', 'S2', 'mert'),
          Trial('07_S2_EAT_fullft', 'S2', 'eat')]


def stable_hash(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def file_hash(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def save_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)


def family(row: AudioRow) -> str:
    parts = row['source'].lower().split('/')
    return parts[1] if parts[0] == 'echoes' else parts[0]


def groups(rows: list[AudioRow]) -> set[str]:
    return {group for row in rows for group in row['group_id'].split('|') if group}


def require_disjoint(split: Partition) -> None:
    for name, rows in split.items():
        if len({r['ID'] for r in rows}) != len(rows):
            raise RuntimeError(f'Duplicate ID in {name}')
        if len({r['pcm_sha256'] for r in rows}) != len(rows):
            raise RuntimeError(f'Duplicate PCM within {name}')
        if any(not r['group_id'] or not r['pcm_sha256'] for r in rows):
            raise RuntimeError(f'Missing lineage/hash in {name}')
    for left, right in [('train', 'tune'), ('train', 'test'), ('tune', 'test')]:
        if groups(split[left]) & groups(split[right]):
            raise RuntimeError(f'Parent/artist leakage: {left}/{right}')
        for key in ['ID', 'audio_sha256', 'pcm_sha256']:
            if {r[key] for r in split[left]} & {r[key] for r in split[right]}:
                raise RuntimeError(f'{key} leakage: {left}/{right}')
    for name in ['train', 'tune', 'test']:
        if {r['MUSIC_FAKE'] for r in split[name] if r['MUSIC_PRESENT'] == '1'} < {'0', '1'}:
            raise RuntimeError(f'Both music classes required in {name}')


def make_splits(locked: Partition) -> dict[str, Partition]:
    # NSynth is excluded in every trial, before any model comparison.
    base = {k: [r for r in rows if family(r) != 'nsynth'] for k, rows in locked.items()}
    require_disjoint(base)
    result = {'S0': base}
    pool = base['train']
    # Transitive connected components, including compound parent IDs.
    parent = {g: g for g in groups(pool)}
    def root(g: str) -> str:
        while parent[g] != g:
            parent[g] = parent[parent[g]]
            g = parent[g]
        return g
    for row in pool:
        tokens = row['group_id'].split('|')
        for token in tokens[1:]:
            parent[root(token)] = root(tokens[0])
    component = {r['ID']: root(r['group_id'].split('|')[0]) for r in pool}
    ordered = sorted(set(component.values()), key=lambda g: stable_hash(['split41', g]))
    count = max(1, round(len(ordered) * .2))
    test_groups, tune_groups = set(ordered[:count]), set(ordered[count:2*count])
    for name, held in [('S1', 'musicgen'), ('S2', 'audioldm')]:
        split = {'train': [], 'tune': [], 'test': []}
        for row in pool:
            group = component[row['ID']]
            partition = 'test' if group in test_groups else ('tune' if group in tune_groups else 'train')
            if partition != 'test' and family(row) == held:
                continue
            split[partition].append(row)
        require_disjoint(split)
        if any(family(r) == held for p in ['train', 'tune'] for r in split[p]):
            raise RuntimeError('Held generator leaked into optimization/selection')
        if not any(family(r) == held for r in split['test']):
            raise RuntimeError(f'No OOD examples for {held}')
        result[name] = split
    return result


def credit_estimate(hours: float, rate: float = 2.98) -> float:
    return round(hours * rate, 2)


def eer_auc(labels: list[int], scores: list[float]) -> tuple[float, float]:
    """Reuse graph_scoring.py's full tied-score ROC / nearest-point EER rule."""
    positives, negatives = sum(labels), len(labels) - sum(labels)
    if min(positives, negatives) == 0:
        raise RuntimeError('EER needs both classes')
    points, tp, fp = [(0., 0.)], 0, 0
    for _, tied in groupby(sorted(zip(scores, labels), reverse=True), key=lambda x: x[0]):
        for _, target in tied:
            tp += target
            fp += 1 - target
        points.append((fp / negatives, tp / positives))
    fpr, tpr = min(points, key=lambda p: abs(p[0] - (1 - p[1])))
    auc = sum((b[0]-a[0])*(b[1]+a[1])/2 for a, b in zip(points, points[1:]))
    return (fpr + (1 - tpr)) / 2, auc

In [ ]:
# 원본 split/기존 고정 voice·presence 예측을 압축 포함: 외부 Drive 링크 불필요
LOCKED_PAYLOAD = '...'
BASELINE_PAYLOAD = 'eNrdfVurrctx3bt+xUHPJnRXX6o7b44jg8jFxnbyEozp658I+e8Z49uGfbz2/KoEEoKzJQ5IaGqtVbO7q8aoy6j/87tffvm/+OeXX37/x//6+//8y+9DDCuHMUIeo8YQr9xUNPbf/823T/39H//7H/7t7//2v/3h3/7xn/7hv+D/Ef6Tamj4aOvy75/53//wx7/7+qEuRfGzW2z//qH/8b/++Y9/9+VDLZZYc6n1P/ygf/ynP/zzH/7nv3z/WGxNc61a/uPP+uFzHT+qhKal4mP/729+NDVpyadknbmlNkI6qw4ZhqnSetOaNFbD1Ka9akrVNLXXFEMP6piapLWI3xm6bWoKoSv+neJnU6WctKO01PNcN+ndO/SslqmaU9UWerJONeLrCCW1aJhaYmk9pRyibWsQyQEXKUXb1pJiSyG3/nKsUrfM2vces51WS5FwRk2GraEI/zxcYcPWgBNLNWjL1g3OKiWobWiTHkLWmqttaMgVP61o7p8NTTGmVuquOPd2i+Cmj96q9VSziOCD9qHi55RSm6hlaCi5hJhjtm1VnL7iXYvYtsKDJNylGPKLrVJHgHlp7hIk395mqfOabgnfT5aQs2Vrh5mht1YMW2vDseeuIo6tmnNJ+IYdv9Twsyrv5ZutK+H73xLzqmtIrOfmEywXDCdRKxxwMc9VG74RvDG1/RL8pdSUnMdacpQae3QcU4+5SYEvTJ9tzbi7cKfzwB/duu/WW/bZhq21pZRaDklNH5xbLq2K+Vh7iDU0fMw51wZXXVrwzjUkeJsIp6mfbS1xSps4hh4mIqHq1HJWMWzFbUKkgw+w7nCpOUiMMVh3uODX1pxaz54TxpGEGoLjm1qHuxT882IqTnTOqWHouGmslNa9zYo3iOdwaEWqFVq1SwipRPNYK6JSFsAD57niQxJUv38lr65J8FjLm6VXU29xSY/466pcBXwKx4o2qtLxWrVZ0aYj8gqedTIvsCCu4mY6h9oK4g3efVTnAsPQ1CVr+Wxry0lDWDfXCfDUWq75bG2WY8KT0NpxW0wnXJVxs9i24lgRJHLyThVBKfVYvIDTEAcr3cRnW0cZq999W8dXUoCfch9tW4+15YxXiOhk4uCOL63gZ4rlhBFdK1xc93AEnDWgRIzeBQ6NL7+/+KUZ95khbcCkfcJqGXa2Fa3YitCP2FrFPNasDVECZ2YCpgIHx7/PtlVADVLGdXePVQkjcVVebC1ygbpFNk71nBDoBbYFDvEN1xRwR61jbcDKSgpkAWE6L8TN7OFD/DI8VXg557m2gtgJpx5enuvOlXQFf78isA4SnDKbhZlalFRobjOvMJ4+UX+1bM1RAeWLeHc4asI9ERcLi8ZMJvHZVgLRnSJiXC2jlRNk57G76ZoQNXOWnsznGqUTRpivVRICIdx5cWgr8Rm4a1IPMsG54gtRfTF17rXh5npZM/YpKkMBYi1TawJ6yQhPlqn4EAgFPIplKxxcFJIS21a86RSBvrJL0bXi+30JrlElyrx51yCdpARUbvRjvVZcAVgaJFk3GMga77WKSVsRzwuIQU4OOiQXhY+TlDw2p/xOEIVfbL33xIGItO7APakDj1eOdawgfzkB9kUT9afcSoFHsV4rfBIeT3HZXKrAmoiH6nkmLXjVeDyfUX9sAfRGxzizNsCnUHCdzxDLC9cMXApjuumZFJipJusGAwt1oJzmOmFwJdCqGD0y11sqeH/ycqxDVoffTVNWBX0FORx9nmFeYZwGI4VlqoCNZDj2YL7WUACqe/CIayvacbBw6y4+xCkIgO6LrTdqmID7ONR7q4xZx94W6k+wVACFLRwBiNYIm0wckRNPFUy+evymBDhC9TAT2XQGs4ry2dS9950jzN5XBTpcfNdxZOsG4zYFfKoX87Ui0CGgW48V7hLOq4h4SFiU8bBlN9yk9uRAPlt6cKK13TvqjgLPtCIuVDUTwnhj+O5sH9wRzjsAQjOzTAI4h8tUHL8EkEkTvpPH93AjDZfpxVLZI63UVj/hRtxL4NvUl8nPA4xN37PaH7FhBzPouZn8vAaYECsAmJf7rgTVJXXXAwNW9/p2fw+QmRxaOWXOI6EUeKVpemD8heAkZu4bUSQLeIkVWBvQI04LH3TSLqni6XcXQ/CYmPzMn6GhhAuqFxRPdMSuGSFTQBHEZK0INIgTWuxgkxvARrSjTS/M9oXms1Y8he+/8RVEwHvBSaQXW5l9XCtt/KQr60qfM8xqpdMKQBZAv5lNYygHGBKToDNnhIfgZ4ThBTuTaX5GOOBVvMQaSWsEXrUdZ57l1hDv7c1MkjITnQU40iStIRBa92RCQzgSvOjueGDA6RJI0v480or/KbQZLwjNOH2UCv804DvNHFPFvcOvtwJr4GFlUEST3ZB04c7l5sJgIT33EsKhBTzBJvFzuJGSZYW8JO28j+SZcrzbJK34YRnRMJuBFagwMGFtvlawQVwR+CaHyYVYCq+Ajw0DfyKwyWdb4Z0RffPpV9strLXO0LOZ/BbcKYT0bJFWVSkPXLYKkIpfjcOS5CaEgeURrD1600oEjsOLfbG1bVBHfBkF/E2rtpRq2NfMCGtRzT1FE/Jrfn6YSW9awaHSDK+wjNdaETM9egNYBWag9c3Uuxa8yCiyF5zYyDkzmWO5pgL3r0xfWFe4AFGRLlW7flNjBkN3jpVpQyJ5L/fdWPLsL5Uq6Xm1OSLzhzjdchB0cOUtcgNQilehdrxBYAWY67lnG0ZkPHq3UCVgDsT8HgxOcL/gyvXFLw34NjYKgEozd8i+iHNMvwSEA6iOUK1mnh9vsFexTa343VGjZ2vDM9WUYnZsDUVyjqwHvNiaC27u3LkAo+voCD4t7vgT9oDIwBFtHGkPC4FwtYG4ObdVQ29wNxX4JdglDVZ4WzPzwUC2eKhAkF75BiFT2OHhXeGAxw8PK58xv8x2A3+ddHioW0a/zPsvM0fa6Vq7WWqFlfiQmPS8pypPftw7VmVhEfjLO1ZcXgT8kj4nmOTsnqbuGydhaa5jk7la6LCWBCL8q6z2x2NlRhBOR80bDIrEWlXyUIS01IJjqCYBYUnlJb0ELwT/kdPZAlAwBg7rnFPN+1sQj4LT19MrQm+NdiatZ3DR3pKTSQO5KKKsyDpnWkqIFQZ/vr4gPfXGq3fz+9eSaWrNZrABRAegs0s3ePatgzn0YpYzSgBFTjV6ZSowJeB1t0yFr62yYeezqYkZ/RQI9W8fbZ4FV7HMakYAfVTATRMs5Qrim+wqFTt/wFezV84AZqk1SIle5YbpoKrxxQMnXFuQbWlLJilcrXEDWFuFVng5PBzcFTO/BKDBZFrrZquABLzpnrtXzgChaniIbi4NP4q2xhdbZ2kyJbR2h4aTwah3mCYyhJcoLOLZeW/8caUHM9jA/2qsybnAjel9BJLgXWAQTLjp8JJfSvmCf4Zz4JYO4BJASenzpp/xsYKx7ga4VPfeCOirwicfM8PfwOEyjqLaqeAcmgCemAnShviAf7p4sabCAgRyh8XBNcTO5HJ7sbWdtlnA0DRTu7vizKbd15NarQB0Jlri2cNLmCQOAR8fgwmOqRIQGmJ1zzRlaUwdfjZUby6A032k0BeABH/x3lZ+tHY+nWBeXwWNTk+fo2VoiQiDOXoQGO8vsN0suj4Jdxf+8uVIl6yFT5y7t4Clg9jOADRhJVxweRFGWlSznNwr+U8X0ymB+8BlBi+Rlll0yt9TxkYqGIYCz3+29UbcX1zZqmONe8JTz+xm3RFHmtkbbJfOe4706haJU7hVOPwmTlwF0o8gXqF7cAk4FI8VLv3FVhFc2PRkWtsC5Vtp72W6JXanNO12woUJmYi/0YLAqYOxdvI4L9pUnEH7jlpe0xDCnDcQ4kdbwWXPYKMpgmq7siNiz71XTLgPSvhj69xXDBFzhGMyMQRYNOCS5t69gjIeTClR3PdKto/n/zm7lJOEG+oca9bRQaaz9F3UQvwAOa0ok5t2J3Rnz2QzaRxwCL+54uGlAoePx+U1B2c2/wZE88+m5gY8WFo5EbCwzKXaN74ZM2eIpwq20cxccGTvestqtn8At4LUZre3Unj+WYvbzF+A5ru+ACZma2+dp7cNrAS4v7KsadbOGyJma8y72NCwNpxrrWaDu0b8aeonIlrgO/TAIatFAcHkc9Ywt6EX0TSwbxl+bgTAiCvFrCjHiDfxtZvsq6msyoBtZrOcAbyGj5Xi9nzjByVNbs83OFcCIW0vth4A/Jo4DpSAv8sFyYnDShsCvbAhrpi5YLCMUhT+yYqugF74u9ix415h9gl+HYL41CbC/Dio92dbu566i4RY2sRhxakiDrthXosgwEQShf1CWcwMf+tsJMwt/hUaBfLGY167VkCSWWcRfH8AYs08VPb9ahW7iZRDAXBMZk9PQjhnWs5Bh2xZz4pg4VzgxykBSupnGJFPGgm/L9wNxLySpDKHmD5Yc870AHYiLcLv52CWWcHfG6hS8RA/Tj4RSHgVql7Z4F5est4lpDP3qG21ucHOBovUYZh9la0iyGnLZtmmPQBHxQysClQVekhuV9rDlUoqblOPsj+svNgaQVWBIuLI+SZZeGC39yn2OBXhpj1h1HP+NmRo4aXM5vxemtcnoMxDiTcNyHEF4UV5sVSAjWBeX/CGR9fFf5FsDhipFnLRr9WWr6ZGPFaEfvMCcwatSfBqrMo6AH5aUbduA6YGd/gZL5V0ypKzW7or1QnWoiPKtJrSHj5SUhULLyleYKC95kArWIuwwOvdYGGevVTvXAWBCwyyh894CRQuHM3gcDFs3MyU5gzRfK2AXjkm/d548vm1slyIoG52MMFJxyrdw4aBlccQ3eb2xgvKutLLudYF/gPXhPu2BuB/Yd1esknlapTEBlyzBY/fRunZrGYg6gsLqPpX6W4HxDgFqHtJXxpvWnrWmsF+r+w3ZJbDGfREvCmlmOdaIqJmbt1zTb2lXKoHmMhtCnuSX86V5wkQ0ca6hBCAJTi3Es3ONHAXEn8LCLdn7oNw0+Y3GjJ+m3eHgVxjS9WtyMWKaK5v7e0gjXBMW2diyerstdPZILBmoQqXCaHd7sJTxEK293l90CwXN89UeOHAasWfM7RQyFLDHkPaxT91b3D10c0RI47Z116qmYzIjXQvd7MmJ6GAbLjdWhzzDDl9nQX+UJOLmgGuXpJMZQAt9cOTH3j/EeFEESfMND+LxYFZJDPxEjQRqtkTgZUDVW5vZUjA5pnTr94NZqdxfZlULgCHHCttd+6JOLsRgfI556djNyyYz5EHs0w1BZxvxgM75thYElbjgw35OWLAFicTReBy0Nl4kbXWihvEZi13TBlstMWXhHAN97Z2mA+GS9K5t8TVmxltIjNwWaKNIkLsDIjWqRaWCkGWu9dZCRNYsfdGFgCVMrB3fxnzrJJLvmHjmsNFzDPyHLA+m6P2ma+6221pMJRIQ8wkKRPaou44ILA6UA6ImteXVh5aBTz32da0JjjBXTv3fRUeedTSg5n8Vp6EZLMxDSiCiZLa1RxFaaTT0UNMTcFN2q+kFgxZASZdXvrbYdjUjoe6Jv64LWP1fU5c9jwgHrl0e0CD6eDI8QbrDhdNzFW5Q3KIlrFRqMTthOawT3yRPKm4HmUqjpXaI7tGmKA7XDsbUSm2Y06gayiBBchuIuFQO4NmdRETWwVjTp4XFnIcOooXW29cLQxWvVos9wCZ1rQdyRME4CZ2mhSsFKRUsomEOTPO7GH5C5XQle07OecXPwx0mDSu2nRlnYPgfwSxihqZGksUAUqmNEalbkuo1nulDkDj0bqyPU/HWXaLkAlQPmuLn9OktTPjnRdOgSP3Ie8BTmSmhGtFaAVcD+ZEIOcyHcEIzT0zge/lXipFUUBw1GtwZ/mpAm5+tnTIBSSU29a6sFTrmh2R0bzBONeIAGDyG7aaUxbJxExP533P4g4ENqBN/FavVpWYnQ3y8lhHk7EXfGrcvJNxdgCLe8zKMvALSJMJJDLQdG7567DM13oWGDU7jZvbR8q0iVvToCwOydfL2EKdtZ0IO6usVeK8pxM+mVSOBW04k+p0p7FL6Ov7+vohiQwj6tlKxMFmJg9IIFI/XYcvtq48ypihSD1L76gTRDZfEyBydJcFNxMg8gOxFLEfK0tabpKJEgaVSXf3Amdq9ry0zNbVDg50ZJls5D9z7QpfV03ZNKWnE9PSoKElNmWY6W9ccE7mdlctgkOtpXtjCyEXwrTPhu46KtxzXhSSKjrhcMJNZmsPXFKQlO2iMthXCNl0v4X1ToIqBwUj5ne8F+kuu6ECSHzJG9ajd996ItAw4BJcDYmf3e5C7MXSjJk2ZKEV8Nass8IbUV2jdlcfjv2NMbojcjnyur10kaoAHs17ez6TWAnMZc/ZzXnWglCkv2pQ+tywFTgt6kwpV6Zvs+uTAsdPGwX9XLCEc6Vy3Wdb056hRxD0XmqA5y2zwXRTmoiqb9KSOilSvGigdJO0Cju0CUk9sMQh9a5aXW2ix8d9zhpqnqnf1cZOk7qkMtjFqmZzMKVpAG9tWReQ4ICXU2ywVCs5fHOa0woFCUVF/PRS55D3Z3pOBAzoGw8AtfZzZePvq8NWJoKbkNQcTQx4L3wbalcfQS5YDPDatfgaevQ6+XFzlVXR+hntK0fYT95hACLtfjr7+MO1x8YeJTwVx9IqOFbbBStlTXN3xRMK+xLV13IMWfLLMCsA7057S9g0NN4QUt6rW4ZSCwXOK0anqafR8SezCxp/O/5+8RQxEGgqp1BbdZt6APRFXvpICQWC9hnYJ5DynBtkRM/6GVXEtAPkz3jhP+4F8j9njh2L+VYBCVk1yM22NXDeykyltQS6/QxuO26J/II9IB7Yb3hafDufTV0Z9HnFNGoDbFoB34yTcXkaudlcac/Z58d5mZ3QzKI9BQi3zkpnA+DiPVbEraT42ItfWgoeXhFr6gVN3n3AKYvpgpNwfD43R5UoPu0d0ezVArTJqXk1qpTAanBDvP72xH5PQO+XyV1dM63ax7qXc4p8tonkwOQ1zzR2taduMjVm2Zps2QpaIXgOMXowuCYpuCUeNQ+pMiXwIn6t64QbxjwTh5p1DFD+k4bJzZlHB6EWR08XyDCVYI7ulgIMnEJ3z7WzOBJ8tZ5CSV3Aw8+2ngyY1OsOQBMd4HYLvGczq1QV/rB/H/X7fIXZzqkqDjXvidlFb+4RAYJfSXUryrkIa3IvloK2wQmO2OeBkRduOLWdzYHsBHQYm9nsEnrghHI3fXAHKgSHLElcdcPOViwvuRTgMjMuyhsMvrJCv6pj4k+bss/C2zY7KzMnISh5YCGmkkKisKWpTEShX55EcssZUaht6b3WQPWPxjn6V1t1jjCognkvBdzj1mm+1lgBdUScuQVm+Xoxr/ADcJOHDTmKT8jc3OIj7m+kiMWLpUPraLMAP2YZF0xnj5Piz9hwCMd2OssTQ9rE+coJm0jRhPzAL8p4bba6FA5f2FInPPmQojt5w3InJa6Se65Sv+mtv9g6R8FPSWUiyiBiDsQ7cHVbPoFDNTE6uquI+xqTWaQS0FHg6uZ4JoJ1PGx14w0QDoebXrww8MWqIYxd2q7wEaWzlcneQYAfpmA50dZdpRgl5zSc/uAENucP2rNfL5Tk6ulqe6SXP6fTWhpMdo8j/OpCHu1QsdzywqBgWUHQbYZDVRfOaajd7KKcgPMWhuDBtJBL8t4rlX6F8nYv7zXdgkPtm52Ht/a57qKSzU+op/uNXcZFMYN94i5seUHEMXNMlNFgy5Y5FIijAEC0TpVdaU8nmTfTWkMFy2jFzab1Z+HC26mW3srQ2dLCL53ACHP1ma+dEKb+vZ1Oo+pyAPk3UT/vOI6ieDOBnKgJiEuegDuIa38mM99sXQWEFT5VQVyjzvKoxZjRtXOertgNwh1QKLdQ7ew3p+3xczzUT4lBMITW3Da8pEy6vuDD1sNIbcwBHjTP4vC7rGIPaTyjshz2tpmrRJhhCsS1yt1DNVcHSSjnPdhu7OZJMz+ZXoTE2ogBQRVIIq+5jtx0R7vLQhLUIqUgabKQBBgptfttITFmLeByuqfS+e2wfqXm+SqoC2edcFVekMReg3ICoa5zV+73pDlnMgXiekVY7NKSWUBPOefUzTv8xCTQA08SmtpluEeeIKkCWiHAIl5/NvXiXOcZuMQ6Lhir9n27KQPCznXcqKpeqpRreWJ1pL4RAqJbQKcgLGevXJXO9qx3ehkKpLxGKDzbQAWoIfdGoIlkz6SwYyvZwP/JlDoJCeqB41+pq98XgWdY3Rokd6IJ/rLPpuLBJ90TfJnzxXNc8LrqzD9W7jRQ8wYXnCk3N3RTyUY54S3RE6vvrD0TvrhaYlSvU3lRd+G9uPjOTj+xbHglsJy9pdtjGhRBjs30wlpZjFC7bVaA0qOXesHzE6IdV7+9Zoro4+v9bGkug3MybdVNeZfEPVu67McaHodiV3C470xjtrdocEQHwUTdDVUJUbin7FcgW2BPwWfH9MiK7zEPR3gXQdMBGTLn7QOTLzWpOZgNSlpYvzdNTVzFxSY2L/XCJlwca/T8Etx0pZTvi6m9H7yqJXXmuMbo5H3muoVOoM5fXGx5l8xaj1lY7ghbFVfYTR7i6Kli5fVWcp8UnPXLBVYEmhOlbFD0W+tBjAVvPvaOQM7B5G5ubSqPCkhy0qScVc2eAwYSfSKX2+sCsEeNlxe81HXjOraxB7zMPIWZl7p6NekNBbo5SWIGmyiZaldmpp+pCOroeBm1DnzWmfDxOyOArV5eapt76mq3LD2jb8DS2pYpk/xbXYwCJjXW2AoQnKJs2ZNrtuxR+99qVQ4+KbaAIxU80zFZkj9zmCmm0h7CFMwJXqp4UvdT7FIVCDr3IxR/rUTjpig3FVEe2aaXGzx1Sojt7AZvkzZiTht5mBkmagoFEGvb1F644tQ0lRNSHJXypu0bF7/F0NzRMfb81de0S4crajvdJbfAB+MCcEXZHLYKCGh1cmroODCKiFpwCZShMTeTvHZDxCxOqyeXslJ2F3D082sdYQ0SwnjiPYirG8eF/2oea2ZXhgCs2Q1bz67GbOL9RiGIX3GW12KrBKbJPc0euMv4aFB8DjcjzXNCnDpSq3ffPC6AiVmW4xoj3LpgYogGLPQMg5pLYIQTAbGkP2HJJe9Sd6WYAiWy3xrxBiBcDuOAwZ9de4h95pGWvcYoxG+i5iZeSllj6ub0mCglNbmE1ZtVls6R0OglhJmLlNc1MIPbt9g0Wxe8+goTwP/KjD/jDPoA6T1zLTCzMwpV0xBc8zaL6M9acK5NttNpYDcIKSY8jM/P6R6QaK0riHdLvu5UZcB+OVb8RX3uwLmxfgdiHXi3LbGlcBHsxlN7r3KnomMRTzmtcJQ+uYtg2JsqrtJsUipgv6n/jT4BB8FwbsU1H4f+dbdg1+VK5e+1K8usG7HFPdp9IOwVitF5rvJABEmSXK3vR5/spW1rzLJvYRoKyP/MUmNY85quqXIlNa5e8PZlV5Acc4hXn+U5P8w9//hcgbwfZXM306+P8Gd5OdcJmiRcTpsmfhr+uDAP0MdPqGs+5rphg8sdfAgkrsgEhB3bFrDUR/LdLODASwfhpjizOYK1TIIOn6IL69DFFyYF5c8vK9/JWPHwNtzDubXM2+6Ba7dHqqQTRorNcQSvApfETKjh+wALEm8Fb6i9VgrieXOt+GmJ3Z2fr/CMqZQ5A7ui191MMU5pzRaG5hA608x2y0D5UX/mK2Cm3py6g3JPYVyleh39DDaZoPTzFZ6yM4LMBJ1j8ruXXcJJ9j45UG88xWZvfK+U3AfHdSY1OLD6dRj4U58wMxtZvEmN3HvJHIF8s7XrTuUqAk/dbanseGa3VYSV7Wf2Tk/Ql2fTmtnp/nT4AOO40tBBWKnqf4JiclSqx3+2NeGB9i2TrTGxLslZVyzmdnuBHaU7teXcJROumzQ9sr8jFI/QAfYp96AHr6iBMy3caPOZ5MzC4vOQvdnGChBRgGROthehF8QJtbuZOOjduee4OLXlVgCZo5vrLxwQ8nSE25PsV33px5sVj6/NfM+8ukMDRmzMb/+MY+hzcP0AyFxmI/iuZycwd/MOw6UrlY/sdRoUDOPmJrs/ghsGaooOGibPUOZWvK3vgSG464uezZylwd5TappcMrHWaDfLMlPgWYD7a89mChw+hyKrdv8sQpyou+EegISrB3vzChvPoD+/mM+2Lt2ZK5rrKau0lfG74zYR4m8W+U+cKM6xgtC1XTPlzuHdk13ESZQrKcXuPsRBUIbbQv5VcKwAr9nbPha5+6gXtz+CFTx8w5/J67whLuqaM8kQBoATPnrMoc/CDJKok5PILAiW7zDy87FWrlqrrko/540U6MSBTVW49pd5nxdbx5gzzXO5eQpoCAArD1tgoDYqYQfnWEtPQMPmfCDxXCQY9nKIcIX12Sbqpoa5ZwK++KOtK1BgS3C0O3IiPR0EnhHs3Z4dsRXR1RlwZe1TnWHe1AsAXXXVQHAKRbqXQ0QIi98k7j/bGgGbNv70xxNrh+WlDG22wAAuS03RYTnM+n8Vxf2UV/uVPMprszBLRzG5y6qYMA141Z/LG0vCWUtjxAHLvnPjgMGyuz2cUkPKdn0O8BVUOIqNmsBecy6h/mX2KxNvBDYfvVxh4cYgJsLHGItu+K6SrildzzhHNUjLNbEEwh22znNV/KxYvX537Uz3pubJzpZnzokCSZ9tBeMOUXGecvPSSV0+ariY5BXINPbqzKN32NmDnRrm+BqVB91SDse1kjs3x+Da6ddfnmuuA2Gnj7M4RLyFa8husRsHHon4ZDcOVE4AJbvJh2340qMXXLm2JnBlqzuIQ79O3dnPpmoce4UQ5micwuI6EdAwczEMSCm3e6tJ1PFwgP1tog5QTYGnrytmPpBXMj/O1bvkNT7o+s3WsXeoN625ZwBvTSvjP1gAUeAjaudjtPf9dEoQmhVmBPNc/S28CNOIOI4HTgxJHJx4eaq6cstcJzdW2ofa7uHMbiZfRAMXAZqSIPBafDTS7ObZoNQp0j9F5AU0Invquorfh8v0UnVdA4/4zLbPzVzBm2/jbKp1ppFNHAAw5lPlonYJ1C0xyVymPHStXmmDDQas9HodpWw/oXrIi1uaZXTghp37jaDBXUZdLd6fcbHRQrwB5Rplb5zBLkUBi8exc6Xs9UjOxlaAF+4tskytNVQaW12hDO7Xjd79bS3wL3uzkx44y00wtLc7zoBzMtnNbxYtbfzdZYQ2EzX/NcO1zl3NPvfCbbdF1ZHrjM+O+2LO4Ehmiqt6Wya4HqvF782Yr7lDEeVaK32xlRsRNR5Aw9jY7z4oIDNseV3OEhTxeFzotjIps1vUjHGVDjnbiojppSICtz2wTfzlWG8ZedajdZR02KjW60rdHCKjKmIpjq2NKeGezb6X8qxHqt4NBlrh1qveHFzI1V2g/PlFyn2HM04UPL2V10m7ndHONIfIAssgiWIpZpdwZPqj2epp3D4FgubJ1nehcldx13IF6iQE6j18thWUEThipjB1c4h5xQUeaR1rVjbSaDDpedH0CBVVu3uWKhm5R1e/5xEhyv40enmWxH6+wZt4ajx8Zudbx0mLhbJtPlZNT9urs1854I4XuwZJ2Tk4V3d901PRoAa+17lFss/W9Bdb20o6ZMU94g7rzNK2s/Ud/iYE6hXZtgrweTB3ZuMNFkFkKl5Rg+OIKRd3gKHn/qzue7E1s8mmyh2BOqy4U5fNTbaqjXzbS292qVVW0rVUsyNPYsa9Cm4dXaiPhO+luHipUfn7RcJyA3O1vsYtF3D/qh6yL3vte+yhBUnmLvQWSoxKN2HW0alqE1zNZLwaLpipbh29Pwiyvp2rtmeMbA8FatprhCkACWZ/BJeHlCq2FnbkeuXkyAAKqC1XCOS/gkr/1rVr4K6fwIUpbN05u8ZsvlZCGLFVhAGDS6S4iQkPU+DUffe2DnNBXSv4sMtu8JcDm7wsDNygFgmMZt/BtXia4xRJpq2NPRmtSLcxk7COWtQWZtIK2hg9FcDGVsziausq+yFLfCNyu0tfMU04YEAXFo25NG4us28rshiU7DQ/XiB79prthOsz5CKeEw5w57hyrkxGAzABcpUXU89kP0E+QBJp3I2LfBHF7Gb3LKz1VQf040EEW5aJKJPbXb3Be2mdGqBeLoKTGkVfFw/vMdruYx98vWke3WuXsNW8wUQlVPBRM7/U4EgY68wsf+PKIK/LMgg3X4WvP+uDIggbbTJC3YupU69yOJAjrUvGWDPmbhY0lN14pdnb0NmbnOyBz04qmimk5J0qgioYuueXKDacItfVvJh6ysRNuvMqLi++l8U+JavUWji32JuajxVsP8eQgpl1AcOEGbl2t22Le1NzDl5kzc/0e3mpP24EUo3j1LOqtjVyX03zTaZjKhx1stu2yF0QCrXYKCJT3el7O5aBmErgnk3HCbN7mdm5FxSxwng2Jm+pgBEAYaA7x9Y8xLkSNBV1ChrcVZVMWws3aTFF7hY0+rP40tVijcBx4W2jxgZ4uAgL8x7OddyQB6W/zUZhrqxIfD6WrZRM58pj8w6zYT6G5tWV2dxBY73UCwVIY+pvEuf71i571n5A4cJICc+112nubW2V629Eu70Bp+CuJ9M36SMPDm7gZUklR/b/eQmJFh81l/Ayr3HC6gueJqx4QOzuGD2cZbK5TOkYIOFiy5xzhAV3uJuD91yjRSP81kPm0v3Fw5SzxFfy+VyP7BiPAOxxe0jN7I++2dQ5ryzLAJ4WZ4gBsaSFFO3NRvnpnfFWarBpAyHW2/YDZwhymvvLHr2TJ8NNjBMhh6uia98p2h3gv9GdGifvWDrlXbIcDfeMRFFxUzuCuhElNVNHjbg0s8enmOGVPaW1RfnLZMC58AEIpr6sf2dcTaobnkTaLLeVO1cyJz85hymwxNxKQDLHWRw7pUYUmZNX1NCCl0ANaLehVPGHxbdOJrZCzHoOXA1XVgEvhhPaVXMgnSozGm2ZXa4tCMBr1W6KDrAieOnDEvmtiJtToxZFbQr88tnWpZft37jAK10cBndrnHJtebzC8Wp70XJLz6ZCUwBcC9vr8bV4NUihyEBq6m9apujh29Tc2ST7vYY90sqLZwF4cu1G4cq52JCT44U5NGeLlGb2bOEtuEulK7WZPJYesrLh4W2g9wqFxEdJEsFb98J/psC72XjIFAKcijmSrom9XdwDbFPXQMWN7Him2hMXpfTiTs1Rzg5v4rMXvnlTWYZLIcNAbB3sExZTwYdqS5SEsZWiVb+JmplFHLaycLDLm66K7J/5vin21QlzkUtsL1M4t8g4fd3ZEUfmSfvC5H5Nlh6ZN/4B1f1QTeV+4mIuu+9cztKp7OzYSoEn/Cj98yTsb+Uc5AT+HXHUgfejI+xuBleAPq4Fs6erCkk4bp15rKUB5dToyScTj+rTT+5uvlRJpbysJrgNbBoYbe2wZ9mSZx3wsWYnSGDbdheTu8bnd7K+YddcOajdvfn79vQIJ3cZJNAmF3C8wMPbcGtPHXvu1DXELXBMw94N89tcs3wHwMglYWljpguzLwhAtPdVscE6i7kKkp14eIvBCTeamUn3dnMBGlLsI3gN0VROxenLy8rAO+Op1AIBrFojguCEs+82c2odtxd/oSmO156smqjZtkVZEyAJcckcdTsRWl0VNYrm1rc1OIC96wLq79rOybrG3lv3sbkcdxbBgdm8FeiFEqCmHCssIDj0urYARktNPQXPLyVqLLN167OtC8z1URYDeY041jPyGtdcmYIIkVPuzsqfRlrVbL8UQVspJu3pR3RGOHfaiNV23pMX0H93oIvR+mxGnaC4Z+G92gt5OVqLMGGXcLi6QFKyt8ulRwhWPGWbxvQh/j5XrJ+NJeWFyt0TUtvx4OmvuOte+wAONyuychFkJ8QyJQ9TVqEUlWWqkDywEdPdUZuflm5vJzriFteOgrh+tvXCW26wmtmldJl7phXF3P9O7TYOJpiYv2VG31bNcFOZdgvyVQz/Ry5X8csKIqa7TQ/36Onhoa2/+9ff/X8vKair'
LOCKED_SHA256 = 'f8be800de2a3f9c4d7b0c0228e68a79a03a468344d938c9e380b32ec8257ac3b'
locked_bytes = zlib.decompress(base64.b64decode(LOCKED_PAYLOAD))
assert hashlib.sha256(locked_bytes).hexdigest() == LOCKED_SHA256
LOCKED = json.loads(locked_bytes)
BASELINE_PREDICTIONS = json.loads(zlib.decompress(base64.b64decode(BASELINE_PAYLOAD)))
SPLITS = make_splits(LOCKED)
for name,split in SPLITS.items():
    print(name, {part:dict(files=len(rows),real=sum(r['MUSIC_FAKE']=='0' and r['MUSIC_PRESENT']=='1' for r in rows),
                         fake=sum(r['MUSIC_FAKE']=='1' and r['MUSIC_PRESENT']=='1' for r in rows),
                         groups=len(groups(rows))) for part,rows in split.items()})
    if name != 'S0':
        held = {'S1':'musicgen','S2':'audioldm'}[name]
        print('held generator test:',held,sum(family(r)==held for r in split['test']))

In [ ]:
import zipfile
import shutil
from contextlib import ExitStack

if RUN_STAGE_INPUTS:
    mapping = json.loads((ASSET_ROOT/'existing_audio_map.json').read_text())
    rows = {row['ID']:row for split in SPLITS.values() for group in split.values() for row in group}
    with ExitStack() as stack:
        archives = {name:stack.enter_context(zipfile.ZipFile(STORAGE_ROOT/name))
                    for name in sorted({mapping[key][0] for key in rows})}
        for identifier, row in rows.items():
            relative = Path('v4_validation') / row['path'].split('/v4_validation/')[-1]
            target = PROJECT_ROOT / relative
            if target.exists():
                if file_hash(target) != row['audio_sha256']:
                    raise RuntimeError(f'Existing audio differs: {target}')
                continue
            archive, member = mapping[identifier]
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix('.partial.wav')
            with archives[archive].open(member) as source, temporary.open('wb') as destination:
                shutil.copyfileobj(source, destination)
            if file_hash(temporary) != row['audio_sha256']:
                raise RuntimeError(f'Extracted audio hash mismatch: {identifier}')
            temporary.replace(target)
    print(f'입력 추출 완료: {len(rows)}개. 전처리·학습은 별도 플래그로 실행합니다.')
else:
    print('입력 추출 미실행: RUN_STAGE_INPUTS=False')


## 5. 원본 검증과 frozen HTDemucs 캐시

원본 파일·PCM 해시를 확인하고 기존 분리 방식(shifts=0, overlap=0.25, 정규화·복원)을 유지합니다. 전체 음원을 분리한 뒤 mono 16kHz 음악 성분에서 구간을 고릅니다. 모델별로 같은 파형을 사용하고, MERT만 입력 직전에 24kHz로 변환합니다.

학습에는 100ms frame 중 20% 이상이 −50dBFS 이상인 구간 후보를 사용합니다. 파일당 최대 8개 후보에서 epoch마다 순서를 유지한 4개를 선택합니다. 후보가 부족하면 같은 구간이 반복될 수 있으며 원본 수가 늘어난 것으로 세지 않습니다. 평가에는 무음 제거 없이 균등 배치한 8개 구간을 씁니다. 5초보다 짧은 음원은 반복해 길이를 맞춥니다. 선택 구간과 유효 여부를 캐시에 기록합니다.

캐시 파일은 파형이므로 full fine-tuning에 사용할 수 있습니다. **고정 MERT 대조군만 embedding cache를 사용**합니다. full FT에 embedding cache를 넘기면 오류로 중단합니다.

In [ ]:
import numpy as np
import soundfile as sf
import torch
import torchaudio
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel


def check_deadline(config) -> None:
    if time.time() >= config['deadline']:
        raise TimeoutError('Session credit/time limit: training stopped; stop the VESSL resource to end billing.')


def audio_path(row: AudioRow, config) -> Path:
    relative = row['path'].split('/v4_validation/')[-1]
    path = Path(config['project_root']) / 'v4_validation' / relative
    if not path.is_file():
        raise FileNotFoundError(f'Mount original v4_validation/audio: {path}')
    return path


def verify_source(row: AudioRow, config) -> tuple[np.ndarray, int]:
    path = audio_path(row, config)
    if file_hash(path) != row['audio_sha256']:
        raise RuntimeError(f'Original file hash mismatch: {row["ID"]}')
    wav, rate = sf.read(path, dtype='float32')
    if rate != 16000 or wav.ndim != 1 or not len(wav) or not np.isfinite(wav).all():
        raise RuntimeError(f'Invalid source audio: {row["ID"]}')
    if hashlib.sha256(wav.astype('<f4').tobytes()).hexdigest() != row['pcm_sha256']:
        raise RuntimeError(f'PCM mismatch: {row["ID"]}')
    return wav, rate


def audio_cache_identity(row: AudioRow, separator_hash: str, config) -> str:
    return stable_hash([row['audio_sha256'], separator_hash, config['separator_checkpoint_sha256'], config['audio_recipe'],
                        config['seconds'], config['pool_windows']])


def validate_audio_cache(row: AudioRow, config) -> None:
    path = Path(config['output'])/'audio_cache'/f'{row["ID"]}.npz'
    record = json.loads(path.with_suffix('.json').read_text())
    expected = audio_cache_identity(row, record['separator_sha256'], config)
    if (record['identity'] != expected or file_hash(path) != record['sha256']
            or record['source_sha256'] != row['audio_sha256']):
        raise RuntimeError(f'Waveform cache content or preprocessing recipe changed: {row["ID"]}')


def prepare_audio(rows: list[AudioRow], config) -> None:
    """Frozen HTDemucs once; preserve baseline normalization/separation settings."""
    from demucs.pretrained import get_model
    from demucs.apply import apply_model
    check_deadline(config)
    # Trusted, hash-checked upstream checkpoint needs legacy pickle loading in PyTorch 2.6+.
    original_load = torch.load
    def trusted_demucs_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_load(*args, **kwargs)
    torch.load = trusted_demucs_load
    try:
        model = get_model('htdemucs', repo=Path(config['model_root'])/'htdemucs')
    finally:
        torch.load = original_load
    model = model.eval().requires_grad_(False).to('cuda')
    model_hash = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        model_hash.update(name.encode())
        model_hash.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    separator_hash = model_hash.hexdigest()
    cache = Path(config['output']) / 'audio_cache'
    cache.mkdir(parents=True, exist_ok=True)
    for row in rows:
        check_deadline(config)
        target, record = cache / f'{row["ID"]}.npz', cache / f'{row["ID"]}.json'
        identity = audio_cache_identity(row, separator_hash, config)
        if target.exists() and record.exists():
            previous = json.loads(record.read_text())
            if previous['identity'] == identity and previous['sha256'] == file_hash(target):
                continue
            raise RuntimeError(f'Stale cache: use a new output directory: {target}')
        raw, rate = verify_source(row, config)
        wav = torch.from_numpy(raw).to('cuda')[None].repeat(model.audio_channels, 1)
        wav = torchaudio.functional.resample(wav, rate, model.samplerate)
        mean, std = wav.mean(0).mean(), wav.mean(0).std()
        if std.item() < 1e-8:
            music = np.zeros(len(raw), dtype=np.float32)
        else:
            with torch.inference_mode():
                sources = apply_model(model, ((wav-mean)/std)[None], device='cuda', shifts=0,
                                      split=True, overlap=.25, progress=False)[0] * std + mean
                music = sources[[i for i,s in enumerate(model.sources) if s != 'vocals']].sum(0).mean(0)
                music = torchaudio.functional.resample(music, model.samplerate, 16000).cpu().numpy()
        width = int(config['seconds'] * 16000)
        starts = list(range(0, max(1, len(music)-width+1), width))
        starts = sorted(set(starts + [max(0, len(music)-width)]))
        valid = []
        for start in starts:
            clip = music[start:start+width]
            frame_count = len(clip)//1600
            active = np.sqrt(np.mean(clip[:frame_count*1600].reshape(-1,1600).astype('float64')**2,1))
            valid.append(bool(frame_count and (active >= 10**(-50/20)).mean() >= .2))
        # Same eight fixed candidates for every encoder; only training uses the activity gate.
        pools, offsets, masks = [], [], []
        for candidates in [np.flatnonzero(valid).tolist() or [0], list(range(len(starts)))]:
            indices = np.linspace(0, len(candidates)-1, config['pool_windows']).round().astype(int)
            selected = [candidates[i] for i in indices]
            clips = []
            for i in selected:
                clip = music[starts[i]:starts[i]+width]
                clip = np.tile(clip, math.ceil(width / max(1,len(clip))))[:width]
                clips.append(clip)
            pools.append(np.stack(clips).astype('float32'))
            offsets.append([starts[i] for i in selected])
            masks.append([valid[i] for i in selected])
        temporary = target.with_suffix('.partial.npz')
        np.savez_compressed(temporary, train=pools[0], evaluation=pools[1],
                            offsets=np.array(offsets), valid=np.array(masks))
        temporary.replace(target)
        save_json(record, {'identity': identity, 'sha256': file_hash(target),
                          'source_sha256': row['audio_sha256'], 'separator_sha256': separator_hash,
                          'usable_train': bool(any(valid)), 'source_samples': len(raw),
                          'unique_train_windows': len(set(offsets[0])), 'quality_gate': '20% active 100ms >= -50dBFS'})
        print('PREPARED', row['ID'], flush=True)


class AudioBags(Dataset):
    """Mutable epoch counter controls reproducible window subsets, shared across encoders."""
    def __init__(self, rows: list[AudioRow], config, training: bool = False, feature_dir: Path | None = None):
        self.rows, self.config, self.training, self.feature_dir = rows, config, training, feature_dir
        self.epoch = 0
    def __len__(self) -> int:
        return len(self.rows)
    def __getitem__(self, index: int):
        row = self.rows[index]
        directory = self.feature_dir or Path(self.config['output'])/'audio_cache'
        with np.load(directory / f'{row["ID"]}.npz') as archive:
            values = archive['train' if self.training else 'evaluation'].copy()
        if self.training:
            seed = int(stable_hash([self.config['seed'], self.epoch, row['ID']])[:16], 16)
            chosen = np.sort(np.random.default_rng(seed).choice(len(values), self.config['train_windows'], replace=False))
            values = values[chosen]
        label = int(row['MUSIC_FAKE']) if row['MUSIC_FAKE'] in ('0','1') else 0
        return torch.from_numpy(values), label


def training_weights(rows: list[AudioRow]) -> torch.Tensor:
    counts = collections.Counter((r['MUSIC_FAKE'], family(r)) for r in rows)
    per_class = collections.Counter(label for label, _ in counts)
    return torch.tensor([.5 / per_class[r['MUSIC_FAKE']] / counts[(r['MUSIC_FAKE'],family(r))]
                         for r in rows], dtype=torch.double)

## 6. 공통 head와 pretrained encoder

head는 기존 `music_temporal.py`의 compact C08 경로를 재사용한 코드입니다. MERT는 마지막 hidden states를 구간 내 평균하고, EAT는 CLS를 제외한 patch tokens를 평균합니다. 둘 다 구간당 1024차원이므로 같은 head에 연결합니다.

EAT 입력은 공식 fbank(128 mel, 16kHz)와 정규화 상수를 사용하며 5초 입력을 512 frame으로 맞춥니다. encoder의 학습 대상 파라미터는 모두 optimizer에 들어갑니다. EAT의 고정 sinusoidal 위치 값과 사용하지 않는 MERT masking token은 학습 대상에서 제외합니다.

첫 실제 backward에서 입력단·첫 Transformer block·마지막 block·head의 gradient를 확인하고 증거를 저장합니다. 이 검사가 통과하지 않으면 full fine-tuning 결과로 진행하지 않습니다.

In [ ]:
class ResidualTemporalBlock(nn.Module):
    """Non-causal temporal convolutions; mask every layer to isolate padding."""

    def __init__(self, dilation: int, dropout: float = .2, *, channels: int = 128,
                 layers: int = 2) -> None:
        super().__init__()
        self.norms = nn.ModuleList([nn.LayerNorm(channels) for _ in range(layers)])
        self.convs = nn.ModuleList([
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation) for _ in range(layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        residual = hidden
        for norm, conv in zip(self.norms, self.convs, strict=True):
            hidden = norm(hidden.transpose(1, 2)).transpose(1, 2).masked_fill(mask, 0)
            hidden = self.dropout(F.gelu(conv(hidden))).masked_fill(mask, 0)
        return (residual + hidden).masked_fill(mask, 0)

class CompactHead(nn.Module):
    """Exact compact C08 MLP path from music_temporal.TemporalMusic; no head search."""
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(1024)
        self.projection = nn.Conv1d(1024,64,1)
        self.blocks = nn.ModuleList([ResidualTemporalBlock(1,.1,channels=64,layers=3)])
        self.output = nn.Sequential(nn.Linear(64,32),nn.GELU(),nn.Dropout(.1),nn.Linear(32,2))
    def forward(self, features: torch.Tensor, padding_mask: torch.Tensor) -> torch.Tensor:
        features = features.masked_fill(padding_mask[...,None],0)
        mask = padding_mask[:,None,:]
        hidden = F.gelu(self.projection(self.input_norm(features).transpose(1,2))).masked_fill(mask,0)
        for block in self.blocks:
            hidden = block(hidden,mask)
        empty = padding_mask.all(1)
        pooled = hidden.masked_fill(mask,-torch.inf).amax(2).masked_fill(empty[:,None],0)
        logits = self.output(pooled)
        return torch.where(empty[:,None],logits.new_tensor([0.,-100.]),logits)

In [ ]:
MODELS = {
    'mert': ('m-a-p/MERT-v1-330M', '5240c2708a5acaee1007f43fb9735c7dcd0b78c9'),
    'eat': ('worstchan/EAT-large_epoch20_pretrain', '1109aaae544915a2b82182c643c1894b1b4d9f27'),
}


class MusicClassifier(nn.Module):
    def __init__(self, encoder: Literal['mert','eat'], frozen: bool, model_root: Path):
        super().__init__()
        self.kind, self.frozen = encoder, frozen
        self.head = CompactHead()  # Identical head initialization before different encoder constructors.
        self.encoder = AutoModel.from_pretrained(model_root/encoder,
            trust_remote_code=True, local_files_only=True)
        if frozen:
            self.encoder.requires_grad_(False)
        if encoder == 'mert':
            self.encoder.config.apply_spec_augment = False
            self.encoder.config.layerdrop = 0.0
            # Only the unused masking token is frozen; convolution + all Transformer layers train.
            if hasattr(self.encoder, 'masked_spec_embed'):
                self.encoder.masked_spec_embed.requires_grad_(False)
        # EAT's fixed sinusoidal positions remain fixed, as in upstream pretraining.

    def encode_windows(self, waves: torch.Tensor) -> torch.Tensor:
        match self.kind:
            case 'mert':
                with torch.autocast(waves.device.type, enabled=False):
                    wav = torchaudio.functional.resample(waves.float(), 16000, 24000)
                    wav = (wav-wav.mean(-1,keepdim=True)) / torch.sqrt(wav.var(-1,keepdim=True,unbiased=False)+1e-7)
                features = self.encoder(input_values=wav).last_hidden_state.mean(1)
            case 'eat':
                mels = []
                for wav in waves:
                    with torch.autocast(waves.device.type, enabled=False):
                        wav = wav.float()
                        mel = torchaudio.compliance.kaldi.fbank((wav-wav.mean())[None],
                            htk_compat=True, sample_frequency=16000, use_energy=False, window_type='hanning',
                            num_mel_bins=128, dither=0., frame_shift=10)
                    # 5 seconds -> pad time to 512 frames, matching official normalization order.
                    mel = F.pad(mel, (0,0,0,max(0,512-len(mel))))[:512]
                    mels.append((mel + 4.268) / (4.569 * 2))
                tokens = self.encoder(torch.stack(mels)[:,None])
                features = tokens[:,1:].mean(1)
            case unknown:
                from typing import assert_never
                assert_never(unknown)
        return features

    def forward(self, bags: torch.Tensor) -> torch.Tensor:
        if bags.shape[-1] == 1024:
            if not self.frozen:
                raise RuntimeError('Cached embeddings cannot train a full encoder')
            features = bags
        else:
            count, windows, length = bags.shape
            if self.frozen:
                self.encoder.eval()
            with torch.set_grad_enabled(self.training and not self.frozen):
                features = self.encode_windows(bags.reshape(-1,length))
            features = features.reshape(count,windows,1024)
        mask = torch.zeros(features.shape[:2], dtype=torch.bool, device=features.device)
        return self.head(features, mask)


def cache_frozen_features(model: MusicClassifier, rows: list[AudioRow], config) -> Path:
    directory = Path(config['output'])/'mert_frozen_features'
    directory.mkdir(exist_ok=True)
    model.eval()
    for row in rows:
        check_deadline(config)
        path = directory/f'{row["ID"]}.npz'
        wave_path = Path(config['output'])/'audio_cache'/f'{row["ID"]}.npz'
        identity = stable_hash([file_hash(wave_path), MODELS['mert'], config['model_manifest_sha256'], config['worker_sha256']])
        record = path.with_suffix('.json')
        if path.exists() and record.exists():
            metadata = json.loads(record.read_text())
            if metadata['identity'] == identity and file_hash(path) == metadata['sha256']:
                continue
            raise RuntimeError(f'Frozen feature cache mismatch: {path}')
        outputs = {}
        with np.load(wave_path) as data, torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
            for key in ['train','evaluation']:
                pieces = []
                for start in range(0,len(data[key]),config['train_windows']):
                    waves = torch.from_numpy(data[key][start:start+config['train_windows']]).cuda()
                    pieces.append(model.encode_windows(waves).float().cpu().numpy())
                outputs[key] = np.concatenate(pieces)
        temporary = path.with_suffix('.partial.npz')
        np.savez_compressed(temporary, **outputs)
        temporary.replace(path)
        save_json(record, {'identity': identity, 'sha256': file_hash(path)})
    return directory


def optimization_groups(model: MusicClassifier, config) -> list[dict]:
    groups_out = []
    for module, lr in [(model.encoder, config['encoder_lr']), (model.head, config['head_lr'])]:
        for decay in [False, True]:
            parameters = [p for name,p in module.named_parameters() if p.requires_grad and
                          (p.ndim >= 2 and not name.endswith('.bias')) == decay]
            if parameters:
                groups_out.append({'params': parameters, 'lr': lr,
                                   'weight_decay': config['weight_decay'] if decay else 0.})
    return groups_out


def lr_factor(step: int, total: int, warmup: int) -> float:
    if step < warmup:
        return (step+1)/max(1,warmup)
    progress = min(1., (step-warmup)/max(1,total-warmup-1))
    return .01 + .99 * .5 * (1 + math.cos(math.pi * progress))


def gradient_evidence(model: MusicClassifier) -> dict[str, float]:
    evidence = {}
    for name, p in model.named_parameters():
        if p.grad is not None:
            if not torch.isfinite(p.grad).all():
                raise RuntimeError(f'Nonfinite gradient: {name}')
            evidence[name] = float(p.grad.detach().abs().sum())
    if not any(v > 0 for k,v in evidence.items() if k.startswith('head.')):
        raise RuntimeError('Classification head is disconnected')
    if model.frozen:
        if any(k.startswith('encoder.') for k in evidence):
            raise RuntimeError('Frozen encoder received gradients')
    else:
        names = [name for name,p in model.encoder.named_parameters() if p.requires_grad]
        if model.kind == 'mert':
            prefixes = ['feature_extractor.', 'encoder.layers.0.',
                        f'encoder.layers.{model.encoder.config.num_hidden_layers-1}.']
        else:
            prefixes = ['model.local_encoder.', 'model.blocks.0.',
                        f'model.blocks.{model.encoder.config.depth-1}.']
        for prefix in prefixes:
            expected = [name for name in names if name.startswith(prefix)]
            if not expected or not any(evidence.get('encoder.'+name,0)>0 for name in expected):
                raise RuntimeError(f'Full fine-tuning disconnected: {prefix}')
    return evidence

## 7. 학습·재개·평가

AdamW, encoder LR 1e−5/head LR 1e−4, warmup 5% 후 cosine decay(최저 1%), weight decay 0.01을 사용합니다. bias와 1차원 파라미터에는 decay를 적용하지 않습니다. L1 및 별도 L2 loss는 추가하지 않습니다.

한 microbatch는 1곡 × 4구간이며 8회 누적해 optimizer step당 8곡/32구간을 처리합니다. gradient clipping은 1.0입니다. 모델별 native masking은 비교 조건을 맞추기 위해 사용하지 않고 추가 waveform augmentation도 첫 비교에는 넣지 않습니다.

checkpoint는 tune Music EER → balanced BCE → 먼저 나온 epoch 순으로 선택합니다. 3 epoch 연속 개선이 없으면 종료합니다. 단일 seed=41 선별 실험이며 작은 EER 차이를 확정적인 우위로 해석하지 않습니다. 매 epoch 모델·optimizer·scheduler·RNG를 저장합니다. 중간 epoch에서 중단되면 그 epoch는 마지막 완료 checkpoint부터 다시 실행합니다.

각 trial의 설정과 선택 checkpoint를 고정한 뒤 평가합니다. 다른 trial 결과를 보고 남은 trial 설정을 바꾸지 않습니다. S0 File EER는 기존 voice/presence 예측을 고정한 결합 결과이며 공식 총점은 계산하지 않습니다. S1·S2는 음악 단독 일반화 평가입니다. 기존 mixture/splice 패널의 생성기 편향은 이번 노트북에서 새 패널을 만들지 않아 남아 있습니다.

In [ ]:
def music_report(rows: list[AudioRow], probabilities: list[float]) -> dict:
    indices = [i for i,r in enumerate(rows) if r['MUSIC_PRESENT']=='1' and r['MUSIC_FAKE'] in ('0','1')]
    labels = [int(rows[i]['MUSIC_FAKE']) for i in indices]
    scores = [float(probabilities[i]) for i in indices]
    result = {'count': len(labels), 'real': labels.count(0), 'fake': labels.count(1), 'eer': None, 'auc': None}
    if set(labels) == {0,1}:
        result['eer'], result['auc'] = eer_auc(labels,scores)
        losses = [-(y*math.log(max(p,1e-7))+(1-y)*math.log(max(1-p,1e-7))) for y,p in zip(labels,scores)]
        result['balanced_bce'] = sum(sum(v for v,y in zip(losses,labels) if y==c)/labels.count(c) for c in [0,1])/2
    return result


def predict(model: MusicClassifier, data: AudioBags, config) -> list[float]:
    loader = DataLoader(data, batch_size=1, shuffle=False, num_workers=0)
    output = []
    model.eval()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        for bags, _ in loader:
            check_deadline(config)
            output.extend(model(bags.cuda()).float().softmax(-1)[:,1].cpu().tolist())
    return output


def atomic_checkpoint(path: Path, value) -> None:
    temporary = path.with_suffix('.partial.pt')
    torch.save(value, temporary)
    temporary.replace(path)


def train_trial(trial: Trial, splits: dict[str, Partition], config) -> None:
    check_deadline(config)
    folder = Path(config['output'])/trial.name
    folder.mkdir(exist_ok=True)
    split = splits[trial.split]
    learning_keys = ['seed','seconds','train_windows','pool_windows','epochs','patience',
                     'encoder_lr','head_lr','weight_decay','warmup_fraction','accumulation','audio_cache_fingerprint']
    identity = stable_hash([trial.__dict__ if hasattr(trial,'__dict__') else
                            [trial.name,trial.split,trial.encoder,trial.frozen], split,
                            {key:config[key] for key in learning_keys}, MODELS, config['model_manifest_sha256'], config['worker_sha256']])
    meta = folder/'identity.json'
    if meta.exists() and json.loads(meta.read_text())['identity'] != identity:
        raise RuntimeError(f'Configuration changed: use another output directory: {folder}')
    save_json(meta, {'identity':identity, 'historical_test':trial.split=='S0',
                     'held_generator': {'S1':'musicgen','S2':'audioldm'}.get(trial.split),
                     'selection':'tune Music EER, balanced BCE, then earliest epoch'})
    if (folder/'complete.json').exists():
        completed = json.loads((folder/'complete.json').read_text())
        if file_hash(folder/'best.pt') != completed['best_checkpoint_sha256']:
            raise RuntimeError('Completed checkpoint changed')
        print('SKIP_COMPLETE',trial.name,flush=True)
        return
    random.seed(config['seed']); np.random.seed(config['seed']); torch.manual_seed(config['seed'])
    torch.cuda.manual_seed_all(config['seed'])
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
    usable = []
    excluded = []
    for row in split['train']:
        record = json.loads((Path(config['output'])/'audio_cache'/f'{row["ID"]}.json').read_text())
        (usable if record['usable_train'] else excluded).append(row)
    if {r['MUSIC_FAKE'] for r in usable} != {'0','1'}:
        raise RuntimeError('Quality gate removed a training class')
    save_json(folder/'training_rows.json', {'included':[r['ID'] for r in usable],
                                         'excluded_low_activity':[r['ID'] for r in excluded]})
    model = MusicClassifier(trial.encoder,trial.frozen,Path(config['model_root'])).cuda()
    feature_dir = None
    if trial.frozen:
        all_rows = {r['ID']:r for p in split.values() for r in p}
        feature_dir = cache_frozen_features(model,list(all_rows.values()),config)
    train = AudioBags(usable,config,True,feature_dir)
    tune = AudioBags(split['tune'],config,False,feature_dir)
    optimizer = torch.optim.AdamW(optimization_groups(model,config))
    steps_per_epoch = math.ceil(len(train)/config['accumulation'])
    total_steps = steps_per_epoch*config['epochs']
    warmup = max(1,math.ceil(total_steps*config['warmup_fraction']))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s:lr_factor(s,total_steps,warmup))
    best_key, stale, start_epoch, history = (math.inf,math.inf), 0, 0, []
    latest = folder/'last.pt'
    if latest.exists():
        state = torch.load(latest,map_location='cpu',weights_only=False)
        if state['identity'] != identity:
            raise RuntimeError('Resume checkpoint identity mismatch')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        start_epoch, best_key, stale, history = state['epoch']+1, tuple(state['best_key']), state['stale'], state['history']
        torch.set_rng_state(state['rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
        random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
        del state
    for epoch in range(start_epoch,config['epochs']):
        if stale >= config['patience']:
            break
        check_deadline(config)
        begun = time.time(); train.epoch = epoch
        sampler = WeightedRandomSampler(training_weights(usable),len(train),replacement=True,
                                        generator=torch.Generator().manual_seed(config['seed']+epoch))
        loader = DataLoader(train,batch_size=1,sampler=sampler,num_workers=0)
        model.train(); optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.
        for index,(bags,labels) in enumerate(loader):
            check_deadline(config)
            group_start = index//config['accumulation']*config['accumulation']
            divisor = min(config['accumulation'],len(loader)-group_start)
            with torch.autocast('cuda',dtype=torch.bfloat16):
                logits = model(bags.cuda())
                loss = F.cross_entropy(logits.float(),labels.cuda())
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite loss')
            (loss/divisor).backward()
            if index == 0 and epoch == start_epoch:
                save_json(folder/'gradient_connectivity.json',gradient_evidence(model))
            loss_sum += float(loss.detach())
            if (index+1)%config['accumulation']==0 or index+1==len(loader):
                nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        probabilities = predict(model,tune,config)
        metrics = music_report(split['tune'],probabilities)
        key = (metrics['eer'],metrics['balanced_bce'])
        if key < best_key:
            best_key,stale = key,0
            atomic_checkpoint(folder/'best.pt', {'model':model.state_dict(),'epoch':epoch,'identity':identity})
            save_json(folder/'selection.json', {'epoch':epoch+1, 'metrics':metrics,'identity':identity,'test_used':False})
            save_json(folder/'tune_predictions.json',[{'ID':r['ID'],'probability':p} for r,p in zip(split['tune'],probabilities)])
        else:
            stale += 1
        entry = {'epoch':epoch+1,'train_loss':loss_sum/len(loader),'tune':metrics,
                 'seconds':time.time()-begun,'learning_rates':scheduler.get_last_lr()}
        history.append(entry); save_json(folder/'history.json',history)
        atomic_checkpoint(latest, {'model':model.state_dict(),'optimizer':optimizer.state_dict(),
            'scheduler':scheduler.state_dict(),'epoch':epoch,'identity':identity,'best_key':best_key,
            'stale':stale,'history':history,'rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
            'python_rng':random.getstate(),'numpy_rng':np.random.get_state()})
        print(trial.name,json.dumps(entry),flush=True)
    save_json(folder/'trained.json', {'identity':identity,'epochs':len(history),
                                    'status':'early_stopped' if stale>=config['patience'] else 'max_epochs'})
    # The checkpoint is sealed before any evaluation. No test-based hyperparameter changes.
    state = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(state['model']); del state
    test = AudioBags(split['test'],config,False,feature_dir)
    probabilities = predict(model,test,config)
    report = {'overall':music_report(split['test'],probabilities), 'subsets':{}, 'official_score':None}
    held = {'S1':'musicgen','S2':'audioldm'}.get(trial.split)
    names = sorted({r['panel'] for r in split['test']})
    subsets = {name:[i for i,r in enumerate(split['test']) if r['panel']==name] for name in names}
    if held:
        subsets['unseen_generator_plus_real'] = [i for i,r in enumerate(split['test']) if r['MUSIC_FAKE']=='0' or family(r)==held]
        subsets['seen_generators_plus_real'] = [i for i,r in enumerate(split['test']) if family(r)!=held]
    else:
        for panel in ['music_seen','music_ood']:
            subsets[panel+'_plus_real'] = [i for i,r in enumerate(split['test']) if r['panel'] in (panel,'music_real')]
    for name, indices in subsets.items():
        report['subsets'][name] = music_report([split['test'][i] for i in indices],[probabilities[i] for i in indices])
    save_json(folder/'test_predictions.json',[{'ID':r['ID'],'probability':p} for r,p in zip(split['test'],probabilities)])
    if trial.split == 'S0':
        baseline = {r['ID']:r for r in config['baseline_predictions']}
        labels, file_scores = [], []
        for row,p in zip(split['test'],probabilities):
            if row['ID'] not in baseline:
                raise RuntimeError('Missing fixed voice/presence prediction')
            fixed = baseline[row['ID']]
            if row['FILE_FAKE'] in ('0','1'):
                labels.append(int(row['FILE_FAKE']))
                file_scores.append(max(fixed['VOICE_PRESENT_PROB']*fixed['VOICE_FAKE_PROB'],fixed['MUSIC_PRESENT_PROB']*p))
        report['known_file_eer'],report['known_file_auc'] = eer_auc(labels,file_scores)
        report['voice_presence'] = 'Fixed historical predictions; voice branch unchanged. No official total.'
    save_json(folder/'evaluation.json',report)
    save_json(folder/'complete.json',{'status':'complete','identity':identity,'finished_at':time.time(),
                                    'best_checkpoint_sha256':file_hash(folder/'best.pt')})
    del model,optimizer
    torch.cuda.empty_cache()

In [ ]:
def worker_main() -> None:
    config = json.loads(Path(os.environ['FULLFT_CONFIG']).read_text())
    if not torch.cuda.is_available() or 'H100' not in torch.cuda.get_device_name(0):
        raise RuntimeError('An H100 worker is required; no CPU/MPS training fallback')
    splits = config['splits']
    stage = os.environ['FULLFT_STAGE']
    try:
        match stage:
            case 'prepare':
                rows = {r['ID']:r for split in splits.values() for rows in split.values() for r in rows}
                prepare_audio([rows[k] for k in sorted(rows)],config)
            case 'train':
                trial = next(t for t in TRIALS if t.name==os.environ['FULLFT_TRIAL'])
                save_json(Path(config['output'])/trial.name/'status.json',{'status':'running','started_at':time.time()})
                train_trial(trial,splits,config)
            case unknown:
                raise RuntimeError(f'Unknown worker stage: {unknown}')
    except TimeoutError as error:
        target = Path(config['output'])/os.environ.get('FULLFT_TRIAL','preparation')/'status.json'
        save_json(target,{'status':'budget_stopped','reason':str(error)})
        raise

if __name__ == '__main__' and os.environ.get('FULLFT_WORKER') == '1':
    worker_main()

## 8. 유료 실행 전 입력·프로토콜 검사

아래 검사는 새 학습을 하지 않습니다. 실제 raw audio 해시 검사는 `RUN_PREPARE` 또는 `RUN_TRAIN`을 켰을 때 수행합니다. GPU 이름만 맞는 것으로 학습 검증을 대신하지 않습니다.

In [ ]:
if RUN_CHECKS:
    # 실제 누수 사례를 주입했을 때 분할 검사가 거부하는지 확인
    import copy
    bad = copy.deepcopy(SPLITS['S0'])
    leaked = dict(bad['train'][0], ID='injected_parent_leak')
    bad['tune'].append(leaked)
    try:
        require_disjoint(bad)
    except RuntimeError as error:
        print('주입한 원본 누수 거부:',error)
    else:
        raise AssertionError('Leakage gate failed')
    for split in SPLITS.values():
        require_disjoint(split)
    assert credit_estimate(16,2.98) == 47.68
    assert abs(lr_factor(0,100,5)-.2)<1e-12
    assert abs(lr_factor(4,100,5)-1)<1e-12
    assert abs(lr_factor(99,100,5)-.01)<1e-12
    assert eer_auc([0,1],[.5,.5])[0] == .5
    print('분할·크레딧·스케줄·동점 EER 검사 완료. GPU 학습 검증과는 별개입니다.')
else:
    print('검사 미실행: RUN_CHECKS=False')


## 9. H100 한 장에 순차 실행

공유 전처리를 GPU 0에서 끝낸 뒤 독립 trial 01~07을 순서대로 한 개씩 실행합니다. 실패하면 다음 실험을 시작하지 않고 로그를 남깁니다. 부분 완료를 전체 완료로 표시하지 않습니다.

한도는 아래 supervisor가 시작한 이후의 경과 시간으로 추정합니다. 대여 시작 시점을 `SESSION_STARTED_AT`에 넣으면 이전 준비 시간도 반영됩니다. 같은 OUTPUT으로 재실행하면 기존 시작 시간을 유지합니다. 리소스를 새로 대여한 경우에만 새 시작 시점을 직접 설정하세요. 크레딧 한도 도달 시 worker를 중단하지만 **리소스 과금은 계속될 수 있으므로 콘솔에서 Stop해야 합니다.**

In [ ]:
import subprocess
import sys


def run_jobs(jobs: list[tuple[str, str]], config) -> None:
    """One isolated CUDA worker; never run trials concurrently."""
    output = Path(config['output'])
    pending, active = list(jobs), {}
    failures = []
    try:
        while pending or active:
            if time.time() >= config['deadline']:
                raise TimeoutError('Session limit reached. STOP the VESSL instance to end billing.')
            for gpu in [0]:
                if gpu in active or not pending:
                    continue
                stage,value = pending.pop(0)
                environment = os.environ.copy()
                environment.update(CUDA_VISIBLE_DEVICES=str(gpu), FULLFT_WORKER='1', FULLFT_STAGE=stage,
                    FULLFT_CONFIG=str(output/'runtime.json'), CUBLAS_WORKSPACE_CONFIG=':4096:8',
                    OMP_NUM_THREADS='4', TOKENIZERS_PARALLELISM='false',
                    HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
                environment['FULLFT_SHARD' if stage=='prepare' else 'FULLFT_TRIAL'] = value
                log = output/f'{stage}_{value}.log'
                with log.open('a') as stream:
                    process = subprocess.Popen([sys.executable,str(output/'worker.py')],env=environment,
                                               stdout=stream,stderr=subprocess.STDOUT)
                active[gpu] = (process,stage,value)
                print(f'GPU {gpu}: {stage} {value} / {log}',flush=True)
            for gpu,(process,stage,value) in list(active.items()):
                code = process.poll()
                if code is not None:
                    del active[gpu]
                    if code:
                        failures.append((stage,value,code))
                        raise RuntimeError(f'Worker failed: {failures}; read its log before resuming')
                    print(f'FINISHED GPU {gpu}: {value}',flush=True)
            elapsed = time.time()-config['session_started_at']
            save_json(output/'session_usage.json',{'elapsed_hours':elapsed/3600,
                'estimated_credits':elapsed/3600*config['credits_per_hour'],
                'account_billing_verified':False,'gpu_resource_stopped':False,
                'pending':len(pending),'active':{str(gpu):item[2] for gpu,item in active.items()}})
            if pending or active:
                time.sleep(5)
    finally:
        for process,_,_ in active.values():
            process.terminate()
        for process,_,_ in active.values():
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        print('Workers ended. VESSL GPU allocation is NOT automatically released.',flush=True)

In [ ]:
import fcntl
import nbformat
import importlib.metadata

if RUN_PREPARE or RUN_TRAIN:
    model_manifest = json.loads((ASSET_ROOT/'model_manifest.json').read_text())
    if model_manifest['revisions'] != {key:value[1] for key,value in MODELS.items()}:
        raise RuntimeError('Pretrained revision mismatch')
    for relative, expected in model_manifest['sha256'].items():
        if file_hash(ASSET_ROOT/'models'/relative) != expected:
            raise RuntimeError(f'Model file hash mismatch: {relative}')
    if torch.cuda.device_count() != 1 or 'H100' not in torch.cuda.get_device_name(0):
        raise RuntimeError('이 노트북은 H100 한 장을 보유한 단일 VESSL 환경용입니다.')
    if TRAIN_CONFIG['seconds'] != 5.0:
        raise RuntimeError('다른 입력 길이는 EAT frontend 설정과 비교 프로토콜을 함께 변경해야 합니다.')
    if TRAIN_CONFIG['pool_windows'] < TRAIN_CONFIG['train_windows'] or TRAIN_CONFIG['train_windows'] < 1:
        raise RuntimeError('Invalid crop counts')
    if min(MAX_SESSION_CREDITS,MAX_SESSION_HOURS,CREDITS_PER_HOUR) <= 0:
        raise RuntimeError('Positive resource limits required')
    notebook = nbformat.read(NOTEBOOK_PATH,as_version=4)
    worker_source = '\n\n'.join(c.source for c in notebook.cells if 'worker' in c.metadata.get('tags',[]))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    with (OUTPUT/'session.lock').open('a') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX | fcntl.LOCK_NB)
        worker = OUTPUT/'worker.py'
        code_hash = hashlib.sha256(worker_source.encode()).hexdigest()
        if worker.exists() and file_hash(worker)!=code_hash:
            raise RuntimeError('Worker code changed. Use a new OUTPUT to preserve old checkpoints.')
        worker.write_text(worker_source)
        previous_runtime = OUTPUT/'runtime.json'
        previous_start = json.loads(previous_runtime.read_text())['session_started_at'] if previous_runtime.exists() else time.time()
        started = SESSION_STARTED_AT if SESSION_STARTED_AT is not None else previous_start
        deadline = started + min(MAX_SESSION_HOURS,MAX_SESSION_CREDITS/CREDITS_PER_HOUR)*3600 - 60
        config = dict(TRAIN_CONFIG, project_root=str(PROJECT_ROOT.resolve()),output=str(OUTPUT.resolve()),
                      splits=SPLITS,baseline_predictions=BASELINE_PREDICTIONS,
                      model_root=str((ASSET_ROOT/'models').resolve()),
                      model_manifest_sha256=file_hash(ASSET_ROOT/'model_manifest.json'),
                      separator_checkpoint_sha256=model_manifest['sha256']['htdemucs/955717e8-8726e21a.th'],
                      session_started_at=started,deadline=deadline,credits_per_hour=CREDITS_PER_HOUR,
                      worker_sha256=code_hash,audio_recipe='baseline_htdemucs_native_to16k_5s_pool8_v1')
        check_deadline(config)
        rows = {r['ID']:r for split in SPLITS.values() for group in split.values() for r in group}
        for row in rows.values():
            check_deadline(config)
            verify_source(row,config)
        save_json(OUTPUT/'runtime.json',config)
        save_json(OUTPUT/'protocol.json',dict(config,source_split_sha256=LOCKED_SHA256,
            versions={name:importlib.metadata.version(name) for name in ['torch','torchaudio','transformers','timm','demucs']},
            gpu_names=[torch.cuda.get_device_name(0)],
            scope='7 configurations, one seed, no separator training, no official total',
            raw_hashes_verified=True))
        if RUN_PREPARE:
            run_jobs([('prepare','all')],config)
        if RUN_TRAIN:
            for row in rows.values():
                validate_audio_cache(row,config)
            config['audio_cache_fingerprint'] = stable_hash([
                file_hash(OUTPUT/'audio_cache'/f'{identifier}.json') for identifier in sorted(rows)])
            save_json(OUTPUT/'runtime.json',config)
            order=range(len(TRIALS))
            run_jobs([('train',TRIALS[i].name) for i in order],config)
else:
    print('준비/학습 미실행: RUN_PREPARE=False, RUN_TRAIN=False')

## 10. 결과 확인

각 trial 디렉터리에 `selection.json`, `gradient_connectivity.json`, `history.json`, `evaluation.json`, `best.pt`, `last.pt`가 저장됩니다. 중단되거나 실패한 trial에는 `complete.json`이 없습니다. 아래 표는 누락된 결과를 0점으로 채우지 않습니다.

S0 점수와 S1·S2 점수는 데이터 구성이 다르므로 절대값끼리 순위를 매기지 말고, **각 분할 내부에서 MERT와 EAT의 차이**를 비교하세요. 신뢰구간과 seed 반복은 이번 7개 구성에 포함하지 않았습니다.

In [ ]:
summary=[]
for trial in TRIALS:
    folder=OUTPUT/trial.name
    if not (folder/'complete.json').exists():
        summary.append(dict(trial=trial.name,status='미완료/미실행'))
        continue
    report=json.loads((folder/'evaluation.json').read_text())
    selection=json.loads((folder/'selection.json').read_text())
    summary.append(dict(trial=trial.name,status='complete',epoch=selection['epoch'],
                        tune_eer=selection['metrics']['eer'],test_eer=report['overall']['eer'],
                        held_eer=report['subsets'].get('unseen_generator_plus_real',{}).get('eer')))
for row in summary:
    print(json.dumps(row,ensure_ascii=False))
if any(row['status']=='complete' for row in summary):
    save_json(OUTPUT/'comparison.json',summary)
print('완료',sum(row['status']=='complete' for row in summary),'/ 7')
print('GPU 리소스 종료 여부는 VESSL 콘솔에서 직접 확인하세요.')

## 출처와 검증 범위

- 프로젝트 입력: `results/methods_a100_20260916/unpacked/locked_split.json`, `baseline_predictions.json`
- 데이터 분석: `results/data_audit_20260915/REPORT.md`
- head: `music_temporal.py`의 compact C08 경로
- 분리: `baseline_original/script.py::separate_voice_and_music`
- 지표: `graph_scoring.py::_roc_metrics`의 동점 처리·nearest-point EER
- [MERT330](https://huggingface.co/m-a-p/MERT-v1-330M), [EAT-large pretrained](https://huggingface.co/worstchan/EAT-large_epoch20_pretrain)
- [VESSL 관리형 크레딧](https://docs.vessl.ai/guides/resources/pricing), [VESSL Cloud 요금](https://vessl.ai/en/pricing)

GPU 학습, H100 메모리 사용량, 처리 속도와 성능 개선은 아직 검증하지 않았습니다. 실제 실행에서 gradient 검사와 첫 epoch 측정값을 확인한 뒤 남은 시간·크레딧을 판단해야 합니다.

제작 시 로컬 확인: 노트북 전체 dry-run, 원본 1,356개 파일·PCM 해시, 기존 head·EER 수치 일치, 축소한 랜덤 초기화 encoder의 gradient·optimizer 연결, 중단 후 재개의 가중치·예측 일치를 확인했습니다. 이는 실제 pretrained H100 학습이나 성능 개선 검증을 대신하지 않습니다.
H100 × 1 및 스토리지 로딩으로 수정한 버전은 사용자 요청에 따라 테스트·dry-run·학습을 실행하지 않았습니다. 위 로컬 확인 기록은 수정 전 H100 × 2 버전에 해당합니다.
